# nn3d ile canli 3D sinir agi gorsellestirmeBu notebook nn3d'nin tamamini bastan sona gosterir:1. Modeli **statik** olarak cizmek2. **Egitim sirasinda canli** izlemek3. Farkli orneklerle **elle kare gondermek**4. Buyuk katmanlarda **sanallastirmayi** ayarlamakHer `nn3d.show(...)` cagrisi tarayicida yeni bir sekme acar. Notebook cekirdegiacik kaldigi surece sunucu yasar; ayrica bir sey yapman gerekmez.

In [ ]:
# nn3d pip ile kurulu degilse depodan kullan
import sys, pathlib
if not any(pathlib.Path(p, "nn3d").is_dir() for p in sys.path):
    sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # TensorFlow gurultusunu kis

import keras
import numpy as np
import nn3d

print("keras", keras.__version__, "| nn3d", nn3d.__version__)

## 1. VeriOzellik **adlarini** bir listede tutmak onemli: nn3d bunlari giris noronlarininyanina yazar ve o anki aktivasyona gore renklendirir. Sonuk gri = sessiz ozellik,turkuaz = yuksek aktivasyon. Adlari vermezsen `in[0]`, `in[1]` gorursun.

In [ ]:
OZELLIKLER = [
    "Tenure", "MonthlyCharges", "TotalCharges", "Contract",
    "PaymentMethod", "InternetService", "OnlineSecurity", "TechSupport",
    "PaperlessBilling", "SeniorCitizen", "Partner", "Dependents",
]

rng = np.random.default_rng(0)
X = rng.normal(size=(3000, len(OZELLIKLER))).astype("float32")

# Ogrenilebilir bir sinyal: boylece egitim sirasinda loss'un gercekten
# dustugunu ve aktivasyonlarin degistigini goruruz.
y = (X[:, 0] * 1.4 - X[:, 3] * 0.9 + X[:, 7] * 0.6 > 0).astype("float32")

X_egitim, X_test = X[:2400], X[2400:]
y_egitim, y_test = y[:2400], y[2400:]
print(X_egitim.shape, y_egitim.mean().round(3))

## 2. ModelSiradan bir Keras modeli. nn3d modeli hicbir sekilde degistirmez; sadecekatmanlarin ciktisini okur.

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(len(OZELLIKLER),)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
], name="churn")

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## 3. Statik gorunum`nn3d.show(...)` agi cizer ve tarayiciyi acar. `sample` verdigimiz icin ilk karehemen gonderilir, ekran bos kalmaz.Tarayicida:| ne | anlami ||---|---|| **turkuaz cizgi** | pozitif agirlik || **kizil cizgi** | negatif agirlik || **cizgi parlakligi** | \|agirlik\| x kaynak noronun aktivasyonu || **nokta buyuklugu/parlakligi** | o noronun aktivasyonu || **kirmizi `[ReLU]`** | katmanin aktivasyon fonksiyonu |Kontroller: **surukle** dondur, **tekerlek** yakinlas, bir **noronun uzerine gel**(butun baglantilari altin sariya doner, gerisi soner), **R** kamerayi sifirla.

In [ ]:
view = nn3d.show(
    model,
    sample=X_test[:1],
    input_labels=OZELLIKLER,
    output_labels=["Kayip Olasiligi"],
)
view.url

## 4. Canli egitim`nn3d.Monitor` bir Keras callback'i. `model.fit(callbacks=[...])` icine koy,gerisi kendiliginden olur: agirliklar degistikce cizgilerin rengi ve parlakligi,metrikler de sag ustteki panel canli guncellenir.**`every` ne ise yarar:** her batch'te ekstra bir ileri yayilim yapmak egitimiolculebilir sekilde yavaslatir. `every=10` on batch'te bir kare gonderir; akicigorunur ve maliyeti ihmal edilebilir.> **Not:** `Monitor` kendi `show()` cagrisini yapar ve bir onceki oturumu> (yukaridaki `view`) kapatir. Ayni anda tek bir sunucu tutmak port sizintisini> onler. Bundan sonra `mon.view` uzerinden devam et.

In [ ]:
mon = nn3d.Monitor(
    X_test[:1],
    every=10,
    input_labels=OZELLIKLER,
    output_labels=["Kayip Olasiligi"],
)

gecmis = model.fit(
    X_egitim, y_egitim,
    epochs=25, batch_size=32,
    validation_split=0.2,
    callbacks=[mon],
    verbose=2,
)

## 5. Elle kare gondermeEgitim bittikten sonra da agi besleyebilirsin. Farkli ornekleri tek tekgonderdiginde **hangi noronlarin hangi girdide atesledigini** gozle takipedebilirsin -- modelin ne "dusundugunu" anlamanin en dolaysiz yolu bu.Asagidaki hucreyi calistirirken tarayici sekmesini acik tut.

In [ ]:
import time

for i in range(20):
    x = X_test[i:i + 1]
    tahmin = float(model.predict(x, verbose=0)[0, 0])
    mon.view.update(x, metrics={"tahmin": tahmin, "gercek": float(y_test[i])})
    time.sleep(0.4)

print("bitti")

## 6. Buyuk katmanlar ve sanallastirma`Dense(3072)` gibi bir katmani 3072 nokta cizmek hem okunmaz hem de oncekikatmanla arasinda milyonlarca kenar demektir; tarayici kilitlenir.nn3d her katmandan esit araliklarla `max_neurons` kadar **temsilci** noron secer;aktivasyonlar ve agirliklar ayni indislere gore kirpilir. Yani ekrandaki hernokta ve her cizgi **gercek** bir noron/agirliktir -- sadece hepsi degil.Bir noronun uzerine geldiginde ipucu kutusu gercek numarayi yazar(`noron 1847 / 3072`), boylece hangi alt kumeye baktigini bilirsin.Daha fazla detay istersen `max_neurons` degerini artir. 32 hala akici; 64uzerinde gorsel lapaya donmeye baslar.

In [ ]:
buyuk = keras.Sequential([
    keras.layers.Input(shape=(256,)),
    keras.layers.Dense(1024, activation="relu"),
    keras.layers.Dense(3072, activation="gelu"),
    keras.layers.Dense(768, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
], name="genis-ag")

buyuk_view = nn3d.show(
    buyuk,
    sample=np.random.rand(1, 256).astype("float32"),
    max_neurons=32,          # varsayilan 16
)
buyuk_view.url

## 7. CNN ornegi (opsiyonel)Konvolusyon katmanlarinda her nokta bir **ozellik haritasi** (kanal) demektir,tek bir piksel degil. Kart uzerinde `32  26x26x32` yazar: 32 kanal, 26x26 uzamsalboyut. Noktanin parlakligi "bu ozellik haritasi ne kadar aktif" anlamina gelir.> Bu hucre Fashion-MNIST'i indirir (~30 MB). Internet yoksa atla.

In [ ]:
(Xg, yg), (Xt, yt) = keras.datasets.fashion_mnist.load_data()
Xg = (Xg[..., None] / 255.0).astype("float32")
Xt = (Xt[..., None] / 255.0).astype("float32")

SINIFLAR = ["Tisort", "Pantolon", "Kazak", "Elbise", "Mont",
            "Sandalet", "Gomlek", "Spor Ayk", "Canta", "Bot"]

cnn = keras.Sequential([
    keras.layers.Input(shape=(28, 28, 1)),
    keras.layers.Conv2D(32, (3, 3), activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
], name="moda-cnn")
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])

cnn_mon = nn3d.Monitor(Xt[:1], every=50, output_labels=SINIFLAR)
cnn.fit(Xg[:12000], yg[:12000], epochs=3, batch_size=64,
        validation_split=0.1, callbacks=[cnn_mon], verbose=2)

## KapatmaCekirdegi kapatinca sunucular zaten olur. Bir sekmeyi erken kapatmak istersen:

In [ ]:
# Ustteki CNN bolumu opsiyonel; atlandiysa cnn_mon hic tanimlanmamis olur.
# Bu yuzden degiskenleri globals() uzerinden yokluyoruz.
for ad in ("view", "buyuk_view"):
    v = globals().get(ad)
    if v is not None:
        v.close()

for ad in ("mon", "cnn_mon"):
    m_ = globals().get(ad)
    if m_ is not None and m_.view is not None:
        m_.view.close()

print("sunucular kapatildi")

---## Duz `.py` betiginde ne degisir?Notebook'ta cekirdek acik kaldigi icin sunucu yasar. Duz bir betikte betikbitince surec de biter ve tarayici sekmesi olur. Bunu onlemek icin sonuna`nn3d.wait()` ekle:```pythonmodel.fit(X, y, callbacks=[nn3d.Monitor(X[:1])])nn3d.wait()   # Ctrl+C ile cik```